# Closing Actuariel — Notebook synthèse

Ce notebook rejoue l'intégralité de la chaîne de clôture en une session interactive.
Pour des explications détaillées de chaque module, consulter `reports/`.

## Sommaire
1. Setup et exécution du pipeline
2. Module 1 — Simulation
3. Module 2 — Provisionnement
4. Module 3 — LAT IFRS 4
5. Module 4 — Boni-mali
6. Module 5 — Compte technique
7. Module 6 — Contrôles automatiques
8. Diagnostic central

## 1. Setup et exécution du pipeline

Une seule fonction `executer_closing()` orchestre les six modules.

In [1]:
import os
import sys
import pandas as pd

# Ajouter src/ au path si pas encore installé
REPO = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO, "src"))

from closing.pipeline import executer_closing

res = executer_closing(REPO, verbose=True)

[M1] Simulation du portefeuille...


    50 000 polices  3 468 sinistres  10 081 flux
[M2] Provisionnement (Chain-Ladder, BF, Mack)...
    Ultimate BF total : 18 940 648 EUR (ecart vs verite : -2.03 %)
[M3] Liability Adequacy Test IFRS 4...
    URR a comptabiliser : 92 795 EUR
[M4] Analyse boni-mali...
    Boni-mali global : -780 387 EUR
[M5] Compte technique...
    Ratio combine global : 96.1%
[M6] Controles de coherence...
    3 alerte(s) sur 6 controles


## 2. Module 1 — Portefeuille simulé

50 000 polices, 5 cohortes 2021–2025. Vérité terrain conservée pour le backtesting.

In [2]:
print(f"Polices    : {len(res['polices']):,}".replace(',', ' '))
print(f"Sinistres  : {len(res['sinistres']):,}".replace(',', ' '))
print(f"Fréquence empirique : {len(res['sinistres'])/len(res['polices']):.3%}")
print(f"\nUltimate vrai par cohorte :")
print(res['ultimate_vrai'].apply(lambda x: f'{x:,.0f} €'.replace(',', ' ')))

Polices    : 50 000
Sinistres  : 3 468
Fréquence empirique : 6.936%

Ultimate vrai par cohorte :
annee_survenance
2021    3 361 654 €
2022    2 767 420 €
2023    5 378 177 €
2024    3 965 069 €
2025    3 859 891 €
Name: ultimate_vrai, dtype: str


In [3]:
print("Triangle des paiements cumulés (M€) :\n")
(res['triangle'] / 1e6).round(2).astype(object).fillna('—')

Triangle des paiements cumulés (M€) :



/tmp/ipykernel_313/3356385579.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  (res['triangle'] / 1e6).round(2).fillna('—')


annee_developpement,0,1,2,3,4
annee_survenance,,,,,
2021,1.31,2.12,2.61,2.94,3.11
2022,1.26,1.91,2.26,2.49,—
2023,1.79,3.1,3.94,—,—
2024,1.64,2.59,—,—,—
2025,1.64,—,—,—,—


## 3. Module 2 — Provisionnement

Backtest des trois méthodes contre la vérité terrain.

In [4]:
synth = res['synthese_prov'].copy()
for col in ['Ultimate_vrai', 'CL_sans_queue', 'CL_avec_queue', 'BF']:
    synth[col] = synth[col].round(0)
synth[['Annee', 'Ultimate_vrai', 'CL_sans_queue', 'CL_avec_queue', 'BF']]

,Annee,Ultimate_vrai,CL_sans_queue,CL_avec_queue,BF
0,2021,3361654.0,3106128.0,3232559.0,3243581.0
1,2022,2767420.0,2630329.0,2737393.0,2827913.0
2,2023,5378177.0,4643653.0,4832666.0,4664056.0
3,2024,3965069.0,3772526.0,3926081.0,3994176.0
4,2025,3859891.0,3866407.0,4023783.0,4210922.0


In [5]:
total_vrai = synth['Ultimate_vrai'].sum()
for methode in ['CL_sans_queue', 'CL_avec_queue', 'BF']:
    total = synth[methode].sum()
    ecart = (total / total_vrai - 1) * 100
    print(f"{methode:<20} : {total/1e6:>6.2f} M€  ({ecart:+.2f} %)")

CL_sans_queue        :  18.02 M€  (-6.79 %)
CL_avec_queue        :  18.75 M€  (-3.00 %)
BF                   :  18.94 M€  (-2.03 %)


## 4. Module 3 — LAT IFRS 4

Décliné par segment : le test global masque la déficience sur Particuliers.

In [6]:
res['lat']

,Segment,UPR_brut,DAC,UPR_net,Best_Estimate,Marge,URR_a_comptabiliser,Resultat_test
0,FLOTTE,1.519279e+06,227891.838000,1.291387e+06,8.680921e+05,423295.014948,0.000000,OK
1,PART,2.083267e+06,312490.059125,1.770777e+06,1.863572e+06,-92795.128095,92795.128095,DEFICIENCY


## 5. Module 4 — Boni-mali

Mali global de 780 k€, concentration sur la cohorte 2023.

In [7]:
res['boni_mali'][['Annee_survenance', 'Ultimate_N1', 'Ultimate_N', 
                   'Boni_mali', 'Variation_pct', 'Nature']]

,Annee_survenance,Ultimate_N1,Ultimate_N,Boni_mali,Variation_pct,Nature
0,2021,2.940776e+06,3.106128e+06,-165352.580000,-5.622754,MALI
1,2022,2.552305e+06,2.630329e+06,-78024.055292,-3.057003,MALI
2,2023,4.225982e+06,4.643653e+06,-417670.876484,-9.883404,MALI
3,2024,3.653186e+06,3.772526e+06,-119339.914514,-3.266735,MALI


## 6. Module 5 — Compte technique

In [8]:
compte = res['compte_technique']
compte[['Segment', 'Primes_acquises', 'Charge_courant', 'Boni_mali',
        'Resultat_net', 'SP_comptable', 'Ratio_combine']]

,Segment,Primes_acquises,Charge_courant,Boni_mali,Resultat_net,SP_comptable,Ratio_combine
0,FLOTTE,3031733.00,1.301549e+06,-280939.905302,621960.666334,0.521975,0.791975
1,PART,4201124.42,2.909373e+06,-502763.075235,-368580.944062,0.812196,1.082196
2,TOTAL,7232857.42,4.210922e+06,-783702.980538,253379.722273,0.690547,0.960547


## 7. Module 6 — Contrôles

In [9]:
res['journal_controles']

,Controle,Statut,Detail
0,C1 - IBNR positif sur toutes les cohortes,OK,IBNR minimum = 137 452 EUR
1,C2 - Reconciliation boni-mali (global vs segme...,OK,Global -780 387 vs segments -783 703 EUR -> ec...
2,C3 - Aucun mali significatif par cohorte,ALERTE,"2 cohorte(s) avec mali > 5% : [2021, 2023]"
3,C4 - Aucun segment en premium deficiency (LAT),ALERTE,1 segment(s) deficient(s) : [np.str_('PART')]
4,C5 - Ratio combine global rentable,OK,Ratio combine = 96.1% (seuil 100%)
5,C6 - Aucun segment en perte technique,ALERTE,1 segment(s) en perte : [np.str_('PART')]


## 8. Diagnostic central

Quatre analyses indépendantes convergent vers le même constat : **le segment Particuliers, cohorte 2023, est simultanément sous-provisionné (M2 hors IC Mack), sous-tarifé (M3 deficiency LAT), en mali significatif (M4 −418 k€) et en perte technique (M5 combiné 108,2 %).**

C'est cette triangulation cohérente qui rend le diagnostic actionnable.